# 🤟 Egyptian Sign Language Recognition — LSTM Model
### Dataset: EGSL (56 كلمة — فيديوهات لغة الإشارة المصرية)

**الخطوات:**
1. Mount Google Drive
2. Install Libraries
3. Extract Keypoints من الفيديوهات
4. بناء وتدريب الـ LSTM
5. Evaluation + Confusion Matrix
6. حفظ الموديل للـ Deployment

> ⚙️ **تأكد إن Runtime → T4 GPU شغال قبل ما تبدأ**

## 📦 Step 1 — Install Libraries

In [ ]:
!pip install mediapipe opencv-python tqdm scikit-learn seaborn -q
print('✅ Libraries installed!')

## 📂 Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ✏️ غيّر المسار ده لمسار الـ dataset عندك على Drive
DATASET_PATH = '/content/drive/MyDrive/dataset - rebirth'

import os
folders = sorted(os.listdir(DATASET_PATH))
print(f'✅ Dataset mounted — {len(folders)} folders found:')
print(folders)

## 🗺️ Step 3 — Franco → Arabic Mapping (56 كلمة)

In [ ]:
FRANCO_TO_ARABIC = {
    '3amel eh'      : 'عامل إيه؟',
    '3ieb'          : 'عيب',
    '3rabia'        : 'عربية',
    '3waz'          : 'عوز',
    'akhoia'        : 'أخويا',
    'alam'          : 'ألم',
    'bhab'          : 'بحب',
    'bkrh'          : 'بكره',
    'bsor3a'        : 'بسرعة',
    'delwaty'       : 'دلوقتي',
    'docktor'       : 'دكتور',
    'efrda'         : 'افردة',
    'ehtrem nfsk'   : 'احترم نفسك',
    'elnhrda'       : 'النهارده',
    'erny wra dhrk' : 'ورّيني ورا ضهرك',
    'fatan'         : 'فطان',
    'fe sark'       : 'في سرك',
    'flos'          : 'فلوس',
    'fokk many'     : 'فك مني',
    'gomma'         : 'جمعة',
    'harmy'         : 'حرامي',
    'hba hba'       : 'هبة هبة',
    'helw awy'      : 'حلو أوي',
    'hta hta'       : 'حتة حتة',
    'kdab'          : 'كداب',
    'khaf'          : 'خاف',
    'khaly balk'    : 'خلي بالك',
    'khatbty'       : 'خطبتي',
    'kolya'         : 'كلية',
    'ktab'          : 'كتاب',
    'mafish'        : 'مفيش',
    'makar'         : 'مكر',
    'manzel'        : 'منزل',
    'mash mashy'    : 'مش ماشي',
    'mashy'         : 'ماشي',
    'meraty'        : 'مراتي',
    'mohands'       : 'مهندس',
    'moshwash'      : 'مشواش',
    'msh 3waz'      : 'مش عوز',
    'oda'           : 'أوضة',
    'sabhl elkheer' : 'صبح الخير',
    'sabhlala'      : 'صبحلالة',
    'sadak'         : 'صدق',
    'sahby'         : 'صاحبي',
    'sbak'          : 'صباك',
    'shar3'         : 'شارع',
    'shbak'         : 'شباك',
    't3ban'         : 'تعبان',
    'tmam'          : 'تمام',
    'whashny'       : 'وحشني',
    'whda whda'     : 'وحدة وحدة',
    'y3takr'        : 'يعتذر',
    'yabky'         : 'يبكي',
    'yadhk'         : 'يضحك',
    'yallhwy'       : 'يلا هوي',
    'yshtm'         : 'يشتم',
}

print(f'✅ Mapping جاهز — {len(FRANCO_TO_ARABIC)} كلمة')
print('\nمثال:')
for k, v in list(FRANCO_TO_ARABIC.items())[:5]:
    print(f'  {k:20} →  {v}')

## ⚙️ Step 4 — Config & Imports

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import json, os
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
#  CONFIG — الفيديو عندك 84 frame، هنشتغل بـ 42
# ─────────────────────────────────────────────
SEQUENCE_LEN  = 42    # نص الـ 84 frame → بيحافظ على الحركة وبيوفر وقت
FEATURES      = 258   # pose(132) + left_hand(63) + right_hand(63)
OUTPUT_DIR    = '/content/processed'
MODEL_DIR     = '/content/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

mp_holistic = mp.solutions.holistic
print(f'✅ Config جاهز — SEQUENCE_LEN = {SEQUENCE_LEN} frames')

## 🦴 Step 5 — Extract Keypoints (MediaPipe)

In [ ]:
def extract_keypoints(results):
    """استخرج 258 keypoint من frame واحد."""
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility]
                     for lm in results.pose_landmarks.landmark]).flatten() \
           if results.pose_landmarks else np.zeros(33 * 4)

    lh = np.array([[lm.x, lm.y, lm.z]
                   for lm in results.left_hand_landmarks.landmark]).flatten() \
         if results.left_hand_landmarks else np.zeros(21 * 3)

    rh = np.array([[lm.x, lm.y, lm.z]
                   for lm in results.right_hand_landmarks.landmark]).flatten() \
         if results.right_hand_landmarks else np.zeros(21 * 3)

    return np.concatenate([pose, lh, rh])  # (258,)


def process_video(video_path, seq_len=SEQUENCE_LEN):
    """اقرأ الفيديو واستخرج sequence من الـ keypoints بـ uniform sampling."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()

    if len(frames) == 0:
        return np.zeros((seq_len, FEATURES))

    # Uniform sampling: اختار seq_len frame موزعين على الفيديو كله
    indices = np.linspace(0, len(frames) - 1, seq_len, dtype=int)

    sequence = []
    with mp_holistic.Holistic(min_detection_confidence=0.5,
                               min_tracking_confidence=0.5) as holistic:
        for idx in indices:
            img = cv2.cvtColor(frames[idx], cv2.COLOR_BGR2RGB)
            img.flags.writeable = False
            results = holistic.process(img)
            sequence.append(extract_keypoints(results))

    return np.array(sequence)  # (42, 258)


print('✅ Functions جاهزة')

## 🔄 Step 6 — Build Dataset (بياخد ~30-40 دقيقة على GPU)

In [ ]:
classes = sorted(os.listdir(DATASET_PATH))
classes = [c for c in classes if os.path.isdir(os.path.join(DATASET_PATH, c))]
label_map = {cls: idx for idx, cls in enumerate(classes)}
NUM_CLASSES = len(classes)

print(f'📂 {NUM_CLASSES} classes found')

# احفظ الـ label map
arabic_label_map = {
    cls: FRANCO_TO_ARABIC.get(cls, cls)
    for cls in classes
}
with open(f'{OUTPUT_DIR}/label_map.json', 'w', encoding='utf-8') as f:
    json.dump({'franco': label_map, 'arabic': arabic_label_map}, f,
              ensure_ascii=False, indent=2)

print('\n🗺️ Class → Arabic Mapping:')
for cls in classes:
    arabic = FRANCO_TO_ARABIC.get(cls, f'[{cls}]')
    print(f'  {label_map[cls]:2d}. {cls:20} → {arabic}')

In [ ]:
# ─── الخطوة دي بتاخد وقت — بتستخرج الـ keypoints من كل الفيديوهات ───
X_list, y_list = [], []
errors = []

for class_name in tqdm(classes, desc='🔄 Processing classes'):
    class_path = os.path.join(DATASET_PATH, class_name)
    label = label_map[class_name]

    video_files = (
        list(Path(class_path).glob('*.mp4')) +
        list(Path(class_path).glob('*.avi')) +
        list(Path(class_path).glob('*.mov')) +
        list(Path(class_path).glob('*.MOV'))
    )

    if not video_files:
        errors.append(f'No videos in: {class_name}')
        continue

    for vf in video_files:
        seq = process_video(str(vf))
        X_list.append(seq)
        y_list.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)

np.save(f'{OUTPUT_DIR}/X.npy', X)
np.save(f'{OUTPUT_DIR}/y.npy', y)

print(f'\n✅ X shape: {X.shape}  →  (samples, frames, features)')
print(f'✅ y shape: {y.shape}')
if errors:
    print(f'⚠️ Errors: {errors}')

## 🔁 Step 7 — Augmentation (يضاعف الـ dataset ×3)

In [ ]:
def augment_noise(seq, std=0.01):
    return seq + np.random.normal(0, std, seq.shape).astype(np.float32)

def augment_flip(seq):
    flipped = seq.copy()
    for i in range(0, 33 * 4, 4):    flipped[:, i]       = 1.0 - flipped[:, i]
    for i in range(0, 21 * 3, 3):    flipped[:, 132 + i] = 1.0 - flipped[:, 132 + i]
    for i in range(0, 21 * 3, 3):    flipped[:, 195 + i] = 1.0 - flipped[:, 195 + i]
    return flipped

def augment_time_warp(seq):
    """تسريع أو تبطيء عشوائي للحركة."""
    T = seq.shape[0]
    factor = np.random.uniform(0.8, 1.2)
    new_T  = int(T * factor)
    indices = np.linspace(0, T - 1, new_T)
    warped = np.array([seq[int(i)] for i in indices])
    # رجّع للـ original length
    final_indices = np.linspace(0, len(warped) - 1, T, dtype=int)
    return warped[final_indices]

# ضاعف الـ dataset
X_aug = np.concatenate([
    X,
    np.array([augment_noise(s) for s in X]),
    np.array([augment_flip(s)  for s in X]),
], axis=0)

y_aug = np.concatenate([y, y, y], axis=0)

np.save(f'{OUTPUT_DIR}/X_aug.npy', X_aug)
np.save(f'{OUTPUT_DIR}/y_aug.npy', y_aug)

print(f'✅ Augmentation خلصت')
print(f'   Original  : {X.shape[0]} samples')
print(f'   Augmented : {X_aug.shape[0]} samples (×3)')

## 📊 Step 8 — Normalize & Split

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Normalize
N, T, F = X_aug.shape
X_flat  = X_aug.reshape(-1, F)
mean    = X_flat.mean(axis=0)
std     = X_flat.std(axis=0) + 1e-8
X_norm  = ((X_aug - mean) / std).astype(np.float32)

np.save(f'{OUTPUT_DIR}/norm_mean.npy', mean)
np.save(f'{OUTPUT_DIR}/norm_std.npy',  std)

# Split  70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y_aug, test_size=0.30, random_state=42, stratify=y_aug)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

y_train_oh = to_categorical(y_train, NUM_CLASSES)
y_val_oh   = to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = to_categorical(y_test,  NUM_CLASSES)

print(f'✅ Data ready')
print(f'   Train : {X_train.shape[0]} samples')
print(f'   Val   : {X_val.shape[0]}   samples')
print(f'   Test  : {X_test.shape[0]}  samples')
print(f'   Shape : {X_train.shape}  →  (samples, {SEQUENCE_LEN} frames, {FEATURES} features)')

## 🧠 Step 9 — Build BiLSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Dropout, BatchNormalization,
    Conv1D, Bidirectional, Concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)

def build_model(seq_len, features, num_classes):
    inp = Input(shape=(seq_len, features), name='input')

    # Multi-scale Conv1D لاستخراج local motion patterns
    c1 = Conv1D(64, 1, activation='relu', padding='same')(inp)
    c2 = Conv1D(64, 3, activation='relu', padding='same')(inp)
    c3 = Conv1D(64, 5, activation='relu', padding='same')(inp)
    x  = Concatenate()([c1, c2, c3])         # (T, 192)
    x  = BatchNormalization()(x)
    x  = Dropout(0.2)(x)

    # Bidirectional LSTM — يقرأ الحركة في الاتجاهين
    x = Bidirectional(LSTM(128, return_sequences=True), name='bilstm_1')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Bidirectional(LSTM(64, return_sequences=False), name='bilstm_2')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Classifier
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = Dropout(0.3)(x)
    x   = Dense(64,  activation='relu')(x)
    out = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inp, outputs=out, name='EGSL_BiLSTM')
    return model

model = build_model(SEQUENCE_LEN, FEATURES, NUM_CLASSES)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 🚀 Step 10 — Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{MODEL_DIR}/egsl_bilstm_best.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=8, min_lr=1e-6, verbose=1),
]

history = model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    batch_size=32,
    epochs=150,
    callbacks=callbacks
)

## 📈 Step 11 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EGSL BiLSTM — Training History', fontsize=14)

axes[0].plot(history.history['accuracy'],     label='Train', color='#2196F3', lw=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#FF5722', lw=2)
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', color='#2196F3', lw=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#FF5722', lw=2)
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png', dpi=150)
plt.show()

## 🧪 Step 12 — Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

loss, acc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f'\n📊 Test Results:')
print(f'   Loss     : {loss:.4f}')
print(f'   Accuracy : {acc*100:.2f}%')

y_pred     = model.predict(X_test, verbose=0)
y_pred_cls = np.argmax(y_pred, axis=1)

# Top-5 Accuracy
top5 = sum(1 for i, t in enumerate(y_test) if t in np.argsort(y_pred[i])[-5:])
print(f'   Top-5    : {top5/len(y_test)*100:.2f}%')

# Per-class Report بالعربي
arabic_names = [FRANCO_TO_ARABIC.get(classes[i], classes[i]) for i in range(NUM_CLASSES)]
print('\n📋 Per-class Report:')
print(classification_report(y_test, y_pred_cls, target_names=arabic_names))

## 🔥 Step 13 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_cls)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

plt.figure(figsize=(22, 20))
sns.heatmap(cm_norm, annot=True, fmt='.1f',
            xticklabels=arabic_names,
            yticklabels=arabic_names,
            cmap='Blues', linewidths=0.3)
plt.title('Confusion Matrix — لغة الإشارة المصرية', fontsize=16)
plt.ylabel('الكلمة الحقيقية')
plt.xlabel('الكلمة المتوقعة')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/confusion_matrix.png', dpi=150)
plt.show()
print(f'💾 Confusion matrix محفوظة')

## 💾 Step 14 — Save Model + TFLite

In [ ]:
import tensorflow as tf

# Keras format
model.save(f'{MODEL_DIR}/egsl_bilstm_final.keras')
print('✅ Keras model saved')

# TFLite (للـ Mobile)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(f'{MODEL_DIR}/egsl_bilstm.tflite', 'wb') as f:
    f.write(tflite_model)
print('✅ TFLite model saved')

# Metadata
metadata = {
    'model_name'   : 'EGSL_BiLSTM',
    'num_classes'  : NUM_CLASSES,
    'sequence_len' : SEQUENCE_LEN,
    'features'     : FEATURES,
    'classes_franco' : classes,
    'classes_arabic' : arabic_names,
    'label_map'    : {str(i): classes[i] for i in range(NUM_CLASSES)},
    'arabic_map'   : {str(i): arabic_names[i] for i in range(NUM_CLASSES)},
}
with open(f'{MODEL_DIR}/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('✅ Metadata saved')

## ☁️ Step 15 — Copy to Google Drive

In [ ]:
import shutil

DRIVE_SAVE_PATH = '/content/drive/MyDrive/EGSL_Model'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# انسخ كل حاجة على Drive عشان ميتمسحش لو الـ session اتقطع
for file in Path(MODEL_DIR).iterdir():
    shutil.copy(str(file), DRIVE_SAVE_PATH)

# كمان الـ processed data
DRIVE_DATA_PATH = '/content/drive/MyDrive/EGSL_Processed'
os.makedirs(DRIVE_DATA_PATH, exist_ok=True)
for file in Path(OUTPUT_DIR).iterdir():
    shutil.copy(str(file), DRIVE_DATA_PATH)

print('✅ كل الملفات اتحفظت على Google Drive!')
print(f'   Models  → {DRIVE_SAVE_PATH}')
print(f'   Data    → {DRIVE_DATA_PATH}')

## 🎯 Step 16 — Quick Inference Test
**اختبر الموديل على فيديو واحد وشوف الـ output بالعربي**

In [ ]:
def predict_sign(video_path, model, classes, arabic_map, mean, std,
                 seq_len=SEQUENCE_LEN, top_k=3):
    """توقع إشارة من فيديو وارجع النتيجة بالعربي."""
    seq = process_video(video_path, seq_len)      # (42, 258)
    seq_norm = ((seq - mean) / std).astype(np.float32)
    seq_input = np.expand_dims(seq_norm, axis=0)  # (1, 42, 258)

    preds = model.predict(seq_input, verbose=0)[0]  # (56,)
    top_indices = np.argsort(preds)[::-1][:top_k]

    print(f'\n🎯 النتيجة:')
    for rank, idx in enumerate(top_indices, 1):
        franco = classes[idx]
        arabic = arabic_map.get(franco, franco)
        conf   = preds[idx] * 100
        bar    = '█' * int(conf // 5)
        print(f'   {rank}. {arabic:15}  ({franco:15})  {conf:5.1f}%  {bar}')

    return classes[top_indices[0]], arabic_map.get(classes[top_indices[0]], '')


# ─── جرب على أي فيديو من الـ dataset ───
# غيّر المسار لأي فيديو عندك
test_video = os.path.join(DATASET_PATH, classes[0],
                           os.listdir(os.path.join(DATASET_PATH, classes[0]))[0])

print(f'🎬 Testing on: {test_video}')
predict_sign(test_video, model, classes, FRANCO_TO_ARABIC, mean, std)

---
## ✅ خلصنا!

| الملف | المكان |
|---|---|
| `egsl_bilstm_best.keras` | أفضل موديل أثناء التدريب |
| `egsl_bilstm_final.keras` | الموديل النهائي |
| `egsl_bilstm.tflite` | للـ Mobile |
| `metadata.json` | labels + arabic mapping |
| `norm_mean/std.npy` | للـ inference |

### الخطوة الجاية
**Streamlit App** → تفتح الويب كام وتتعرف على الإشارة real-time بالعربي 🎥